# Phase 0 — Retroactive Validation

**Question:** Does an MP + order-flow feature set, scored by LightGBM, reliably flag the worst losers in your FY25-FY26 Zerodha trade log as "skip"?

**Pass criteria:** see [../docs/DECISION_GATE.md](../docs/DECISION_GATE.md).

**Workflow:**
1. Load trade log (drop into `data/raw/zerodha_tradebook_fy25_fy26.csv`).
2. Confirm tick/book data is available for the trade window.
3. Build features → `data/processed/features.parquet`.
4. Build labels (net_R after costs) → `data/processed/labels.parquet`.
5. Walk-forward train LightGBM baseline.
6. Compute skip-accuracy + profit factor + max drawdown.
7. Generate the go/no-go report.

In [ ]:
%load_ext autoreload
%autoreload 2

import pandas as pd
from pathlib import Path

from sniper_phase0.utils.settings import Settings
from sniper_phase0.data.trade_log import load_trade_log

settings = Settings.load('../configs/base.yaml')
settings.paths

## 1. Load and inspect the trade log

In [ ]:
trades = load_trade_log(settings.paths.trade_log)
print(f'{len(trades)} trades from {trades["entry_ts"].min()} to {trades["entry_ts"].max()}')
trades.head()

In [ ]:
trades.groupby('instrument_type').agg(n=('trade_id', 'count'), gross=('gross_pnl', 'sum'))

## 2. Build features (CLI: `phase0 features`)

Run from the project root:

```bash
uv run phase0 features --config configs/base.yaml
```

Or inline below.

In [ ]:
from sniper_phase0.features.build import build_features
features_df, availability_df = build_features(trades, settings)
print(features_df.shape)
features_df.head()

## 3. Labels (CLI: `phase0 label`)

In [ ]:
from sniper_phase0.labels.cost_model import net_pnl
import numpy as np

rows = []
for _, t in trades.iterrows():
    gross, net, _ = net_pnl(
        entry_price=float(t['entry_price']), exit_price=float(t['exit_price']),
        qty=int(t['qty']), side=t['side'], costs=settings.costs,
    )
    stop_distance = max(1e-6, abs(t['entry_price']) * settings.labeling.default_stop_pct / 100.0)
    rows.append({
        'trade_id': int(t['trade_id']),
        'outcome': 'actual',
        'exit_ts': t['exit_ts'],
        'exit_price': float(t['exit_price']),
        'gross_R': gross / (stop_distance * t['qty']),
        'net_R': net / (stop_distance * t['qty']),
        'mae': np.nan, 'mfe': np.nan,
    })
labels_df = pd.DataFrame(rows)
labels_df.describe()

## 4. Walk-forward training

In [ ]:
from sniper_phase0.evaluation.walk_forward import run_walk_forward
folds = run_walk_forward(features_df, labels_df, settings)
len(folds)

## 5. Go/no-go report

In [ ]:
from sniper_phase0.evaluation.reports import build_report, write_report
report = build_report(folds, settings)
report['gate']

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

all_preds = pd.concat([f.predictions for f in folds], ignore_index=True)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.histplot(all_preds, x='p_win', hue=(all_preds['net_R'] > 0), bins=30, ax=axes[0])
axes[0].set_title('p_win distribution by realised outcome')
all_preds.assign(cum_R=all_preds.sort_values('trade_id')['net_R'].cumsum()).plot(
    x='trade_id', y='cum_R', ax=axes[1]
)
axes[1].set_title('Cumulative net_R, all folds concatenated')
plt.tight_layout(); plt.show()

In [ ]:
report_path = write_report(report, settings.paths.reports_out)
print(f'Report written to: {report_path}')
print(f'Phase 0 pass: {report["gate"]["phase0_pass"]}')